# CMIP6 Model Precipitation Extraction and Aggregation
## Reference Period: 1995–2014

**Purpose:**  
Extract daily precipitation from bias-corrected RICCAR CMIP6 regional climate model outputs at observed station locations across Jordan's hydrological basins, and aggregate to monthly and long-term monthly climatologies for the evaluation period 1995–2014.

**Models:** CMCC-CM2-SR5 · CNRM-ESM2-1 · EC-Earth3-Veg · IPSL-CM6A-LR · MPI-ESM1-2-LR · NorESM2-MM  
**Scenarios:** SSP2-4.5  
**Spatial resolution:** 0.1° × 0.1° (SMHI-HCLIM-ALADIN-38, bias-corrected via SMHI-MIdAS021)  
**Variable:** `prAdjust` (bias-adjusted precipitation flux, mm/day)

**Workflow:**
1. Load station locations from the basin mapping file  
2. For each model NetCDF file, find the nearest grid point to each station  
3. Extract the daily time series for 1995–2014  
4. Aggregate to monthly totals (mm/month)  
5. Compute long-term monthly means (climatology) across the 20-year period  
6. Save all outputs in structured folders

**Citation of data source:**  
RICCAR (Regional Initiative for the Assessment of Climate Change Impacts on Water Resources and Socio-Economic Vulnerability in the Arab Region). Data portal: https://www.riccar.org/search-netcdf-listings


## 1. Import Libraries

All required libraries are part of the standard scientific Python stack. Install any missing packages with:
```bash
pip install numpy pandas xarray netcdf4 scipy openpyxl tqdm
```


In [3]:
import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
from scipy.spatial import cKDTree
from pathlib import Path
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

print("Libraries loaded successfully.")
print(f"  numpy   : {np.__version__}")
print(f"  pandas  : {pd.__version__}")
print(f"  xarray  : {xr.__version__}")


Libraries loaded successfully.
  numpy   : 2.1.3
  pandas  : 2.2.3
  xarray  : 2025.12.0


## 2. Configuration

All file paths and analysis parameters are defined here. Update these paths if your directory structure differs.

**Evaluation period:**  
1995–2014 (20 years) — aligned with IPCC AR6 reference period practice (Gutiérrez et al., 2021) and consistent with the CMIP6 historical simulation end date.


In [4]:
# ── Input paths ──────────────────────────────────────────────────────────────
# Station locations and basin assignments
STATION_MAPPING_FILE = (
    r"D:\RICAAR\Pr.New.Stations.Selection\OBSERVATIONS"
    r"\Station_Basin_Mapping.xlsx"
)

# Directory containing the 6 merged model NetCDF files
MODEL_DIR = (
    r"D:\RICAAR\riccar-data_jordan-ssp2-4-5-daily-data_2024-07-29_0915"
    r"\Merge\Precipitation"
)

# ── Output root ───────────────────────────────────────────────────────────────
OUTPUT_ROOT = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments")

# ── Analysis period ───────────────────────────────────────────────────────────
PERIOD_START = 1995   # inclusive
PERIOD_END   = 2014   # inclusive

# ── Model names (must match NetCDF filenames without .nc) ────────────────────
MODELS = [
    "CMCC-CM2-SR5",
    "CNRM-ESM2-1",
    "EC-Earth3-Veg",
    "IPSL-CM6A-LR",
    "MPI-ESM1-2-LR",
    "NorESM2-MM",
]

# ── NetCDF variable name ──────────────────────────────────────────────────────
PR_VAR = "prAdjust"   # bias-adjusted precipitation (mm/day)

# ── Output sub-folder names ──────────────────────────────────────────────────
DAILY_DIR    = OUTPUT_ROOT / "daily_data"
MONTHLY_DIR  = OUTPUT_ROOT / "monthly_data"
LTMM_DIR     = OUTPUT_ROOT / "long_term_monthly_mean"

for d in [DAILY_DIR, MONTHLY_DIR, LTMM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"  Evaluation period : {PERIOD_START}–{PERIOD_END}")
print(f"  Models            : {len(MODELS)}")
print(f"  Output root       : {OUTPUT_ROOT}")
print(f"  Daily output      : {DAILY_DIR}")
print(f"  Monthly output    : {MONTHLY_DIR}")
print(f"  LTMM output       : {LTMM_DIR}")


Configuration loaded.
  Evaluation period : 1995–2014
  Models            : 6
  Output root       : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments
  Daily output      : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\daily_data
  Monthly output    : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\monthly_data
  LTMM output       : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\long_term_monthly_mean


## 3. Load Station Locations

Station locations and basin assignments are read from `Station_Basin_Mapping.xlsx`. The file contains 49 selected precipitation stations across Jordan's 12 hydrological basins with their geographic coordinates (decimal degrees, WGS84).


In [5]:
stations = pd.read_excel(STATION_MAPPING_FILE)
stations["Station_ID"]   = stations["Station_ID"].astype(str).str.strip()
stations["Station_Name"] = stations["Station_Name"].astype(str).str.strip()
stations["Basin"]        = stations["Basin"].astype(str).str.strip()

# Validate required columns
required_cols = {"Station_ID", "Station_Name", "Latitude", "Longitude", "Basin"}
missing = required_cols - set(stations.columns)
if missing:
    raise ValueError(f"Missing columns in mapping file: {missing}")

print(f"Stations loaded : {len(stations)}")
print(f"Unique basins   : {stations['Basin'].nunique()}")
print()
print(stations[["Station_ID", "Station_Name", "Basin",
                "Latitude", "Longitude"]].to_string(index=False))


Stations loaded : 49
Unique basins   : 12

Station_ID                        Station_Name                 Basin  Latitude  Longitude
    AB0004                      KH.EL-WAHADNEH               N.R.S.W 32.328239  35.643545
    AD0002                              HARTHA      YARMOUK (JORDAN) 32.692571  35.844729
    AD0005                             UM QEIS      YARMOUK (JORDAN) 32.654533  35.679241
    AD0008                              KHARJA      YARMOUK (JORDAN) 32.658084  35.887122
    AD0011                         EN NUEIYIME      YARMOUK (JORDAN) 32.417193  35.911881
    AD0023                     JABER MUGHAYYIR      YARMOUK (JORDAN) 32.509924  36.199947
    AD0032     BAQURA MET.STATION (METEO DEPT) JORDAN VALLY (JORDAN) 32.612437  35.596982
    AE0003                           KAFR YUBA               N.R.S.W 32.543115  35.798953
    AE0004                           KAFR ASAD               N.R.S.W 32.600307  35.710913
    AH0002                       WADI EL-YABIS JORDAN VAL

## 4. Nearest Grid-Point Identification

For each station, we identify the nearest model grid cell using a **k-d tree** spatial index (Bentley, 1975) built on the flattened 2-D latitude–longitude grid. This is a standard approach for station-to-grid matching in regional climate model evaluation studies (e.g., Enyew et al., 2024; Azad & Ahmadi, 2024).




In [6]:
def build_kdtree(lats: np.ndarray, lons: np.ndarray):
    """
    Build a k-d tree from a 2-D lat/lon grid.

    Parameters
    ----------
    lats : 1-D array of latitudes  (shape: n_lat)
    lons : 1-D array of longitudes (shape: n_lon)

    Returns
    -------
    tree       : cKDTree object
    grid_coords: (N, 2) array of (lat, lon) for every grid point
    grid_idx   : (N, 2) array of (lat_idx, lon_idx) for every grid point
    """
    lon_grid, lat_grid = np.meshgrid(lons, lats)          # shape: (n_lat, n_lon)
    grid_coords = np.column_stack([
        lat_grid.ravel(),
        lon_grid.ravel()
    ])
    lat_idx, lon_idx = np.unravel_index(
        np.arange(len(grid_coords)), lat_grid.shape
    )
    grid_idx = np.column_stack([lat_idx, lon_idx])
    tree = cKDTree(grid_coords)
    return tree, grid_coords, grid_idx


def find_nearest_grid(tree, grid_idx, station_lat: float, station_lon: float):
    """
    Return the (lat_idx, lon_idx) of the nearest grid cell.

    Parameters
    ----------
    tree        : cKDTree built from build_kdtree()
    grid_idx    : index array from build_kdtree()
    station_lat : station latitude  (decimal degrees)
    station_lon : station longitude (decimal degrees)

    Returns
    -------
    lat_i  : latitude  index into the model grid
    lon_i  : longitude index into the model grid
    dist_km: approximate distance to nearest centre (km)
    """
    dist_deg, idx = tree.query([station_lat, station_lon])
    lat_i, lon_i  = grid_idx[idx]
    # approximate km: 1° latitude ≈ 111.32 km
    dist_km = dist_deg * 111.32
    return int(lat_i), int(lon_i), round(dist_km, 2)


print("Nearest grid-point functions defined.")


Nearest grid-point functions defined.


## 5. Grid-Point Matching — Quality Control

Before extraction, we verify the station-to-grid matching using the first model file as representative (all models share the same grid: 85 lat × 75 lon, 0.1° resolution, 33.55°–40.95°E, 27.05°–35.45°N).

All stations within Jordan's borders should fall well within this threshold given the 0.1° (≈ 11 km) grid resolution.


In [7]:
DISTANCE_THRESHOLD_KM = 25.0  # flag stations beyond this distance

# Open the first model to get the grid (all models share the same grid)
first_model_path = Path(MODEL_DIR) / f"{MODELS[0]}.nc"
with xr.open_dataset(first_model_path) as ds:
    lats = ds["lat"].values
    lons = ds["lon"].values

print(f"Model grid loaded from: {MODELS[0]}.nc")
print(f"  Latitude  : {lats.min():.2f}° to {lats.max():.2f}°N  ({len(lats)} points)")
print(f"  Longitude : {lons.min():.2f}° to {lons.max():.2f}°E  ({len(lons)} points)")
print()

# Build k-d tree once (reused for all models — same grid)
tree, grid_coords, grid_idx = build_kdtree(lats, lons)

# Match every station
qc_rows = []
for _, stn in stations.iterrows():
    lat_i, lon_i, dist_km = find_nearest_grid(
        tree, grid_idx, stn["Latitude"], stn["Longitude"]
    )
    qc_rows.append({
        "Station_ID"    : stn["Station_ID"],
        "Station_Name"  : stn["Station_Name"],
        "Basin"         : stn["Basin"],
        "Stn_Lat"       : round(stn["Latitude"],  6),
        "Stn_Lon"       : round(stn["Longitude"], 6),
        "Grid_Lat"      : round(lats[lat_i], 4),
        "Grid_Lon"      : round(lons[lon_i], 4),
        "Lat_Idx"       : lat_i,
        "Lon_Idx"       : lon_i,
        "Distance_km"   : dist_km,
        "Flag"          : "⚠ DISTANT" if dist_km > DISTANCE_THRESHOLD_KM else "✓ OK",
    })

qc_df = pd.DataFrame(qc_rows)

n_flagged = (qc_df["Flag"] != "✓ OK").sum()
print(f"Stations matched  : {len(qc_df)}")
print(f"Flagged (> {DISTANCE_THRESHOLD_KM} km) : {n_flagged}")
print()
print(qc_df[["Station_ID", "Station_Name", "Basin",
             "Grid_Lat", "Grid_Lon", "Distance_km", "Flag"]].to_string(index=False))


Model grid loaded from: CMCC-CM2-SR5.nc
  Latitude  : 27.05° to 35.45°N  (85 points)
  Longitude : 33.55° to 40.95°E  (75 points)

Stations matched  : 49
Flagged (> 25.0 km) : 0

Station_ID                        Station_Name                 Basin  Grid_Lat  Grid_Lon  Distance_km Flag
    AB0004                      KH.EL-WAHADNEH               N.R.S.W     32.35     35.65         2.53 ✓ OK
    AD0002                              HARTHA      YARMOUK (JORDAN)     32.65     35.85         4.78 ✓ OK
    AD0005                             UM QEIS      YARMOUK (JORDAN)     32.65     35.65         3.29 ✓ OK
    AD0008                              KHARJA      YARMOUK (JORDAN)     32.65     35.85         4.23 ✓ OK
    AD0011                         EN NUEIYIME      YARMOUK (JORDAN)     32.45     35.95         5.60 ✓ OK
    AD0023                     JABER MUGHAYYIR      YARMOUK (JORDAN)     32.55     36.15         7.13 ✓ OK
    AD0032     BAQURA MET.STATION (METEO DEPT) JORDAN VALLY (JORDAN)    

## 6. Save Grid-Point Matching Table

The matching table is saved as a CSV for reproducibility and reporting.


In [8]:
qc_csv = OUTPUT_ROOT / "station_grid_matching.csv"
qc_df.to_csv(qc_csv, index=False)
print(f"Grid-matching table saved: {qc_csv}")


Grid-matching table saved: C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\station_grid_matching.csv


## 7. Extract Daily Precipitation — All Models and Stations

For each of the six CMIP6 models, the following steps are performed:

1. **Open** the merged NetCDF file (1961–2070, variable `prAdjust`, mm/day)  
2. **Slice** the time dimension to the evaluation period 1995–2014  
3. **Extract** the value at the matched grid cell for each of the 49 stations  
4. **Save** a per-model CSV to `daily_data/<MODEL_NAME>/daily_pr_all_stations.csv`

Each output CSV has columns: `Date`, `Station_ID`, `Station_Name`, `Basin`, `pr_mm_day` (daily precipitation in mm/day).

> **Memory note:** Each NetCDF file is ~982 MB. The extraction loads only the required time slice and grid points — the full array is never loaded into memory.


In [9]:
# Pre-build station list for extraction
stn_list = qc_df[["Station_ID","Station_Name","Basin",
                   "Lat_Idx","Lon_Idx"]].to_dict("records")

for model in MODELS:
    nc_path    = Path(MODEL_DIR) / f"{model}.nc"
    model_dir  = DAILY_DIR / model
    model_dir.mkdir(parents=True, exist_ok=True)
    out_csv    = model_dir / "daily_pr_all_stations.csv"

    if out_csv.exists():
        print(f"[SKIP] {model} — daily CSV already exists.")
        continue

    print(f"[PROCESSING] {model} ...")

    with xr.open_dataset(nc_path) as ds:
        # Slice to evaluation period (keep Oct of previous year for wet-season
        # completeness; daily file retains full calendar years)
        ds_period = ds[PR_VAR].sel(
            time=slice(f"{PERIOD_START}-01-01", f"{PERIOD_END}-12-31")
        )
        dates = pd.to_datetime(ds_period["time"].values)
        print(f"  Time slice   : {dates[0].date()} → {dates[-1].date()}  "
              f"({len(dates):,} days)")

        rows = []
        for stn in tqdm(stn_list, desc=f"  Stations", leave=False):
            # Extract time series at this grid point
            ts = ds_period.isel(
                lat=int(stn["Lat_Idx"]),
                lon=int(stn["Lon_Idx"])
            ).values.astype(float)         # shape: (n_time,)

            for d, v in zip(dates, ts):
                rows.append({
                    "Date"         : d.date(),
                    "Station_ID"   : stn["Station_ID"],
                    "Station_Name" : stn["Station_Name"],
                    "Basin"        : stn["Basin"],
                    "pr_mm_day"    : round(float(v), 4) if not np.isnan(v) else np.nan,
                })

    daily_df = pd.DataFrame(rows)
    daily_df["Date"] = pd.to_datetime(daily_df["Date"])
    daily_df.sort_values(["Station_ID","Date"], inplace=True)
    daily_df.to_csv(out_csv, index=False)
    print(f"  Saved → {out_csv}  ({len(daily_df):,} rows)")

print()
print("Daily extraction complete.")


[SKIP] CMCC-CM2-SR5 — daily CSV already exists.
[SKIP] CNRM-ESM2-1 — daily CSV already exists.
[SKIP] EC-Earth3-Veg — daily CSV already exists.
[SKIP] IPSL-CM6A-LR — daily CSV already exists.
[SKIP] MPI-ESM1-2-LR — daily CSV already exists.
[SKIP] NorESM2-MM — daily CSV already exists.

Daily extraction complete.


## 8. Aggregate Daily to Monthly Precipitation

Monthly precipitation totals (mm/month) are computed by summing all daily values within each calendar month. This follows the temporal aggregation convention in Equations (1)–(2) of the manuscript:

$$\text{MS}(y, m) = \sum_{d=1}^{n} P(y, m, d)$$

$$\text{LTMA}(m) = \frac{1}{Y} \sum_{y=1}^{Y} \text{MS}(y, m)$$

where $P$ is daily precipitation (mm/day), $y$ is year, $m$ is month, $d$ is day index, and $Y$ is the number of years in the analysis period.

Each monthly CSV is saved to `monthly_data/<MODEL_NAME>/monthly_pr_all_stations.csv` with columns: `Year`, `Month`, `Station_ID`, `Station_Name`, `Basin`, `pr_mm_month` (monthly total, mm/month).


In [10]:
from tqdm import tqdm

for model in tqdm(MODELS, desc="Processing models"):
    daily_csv   = DAILY_DIR   / model / "daily_pr_all_stations.csv"
    monthly_dir = MONTHLY_DIR / model
    monthly_dir.mkdir(parents=True, exist_ok=True)
    monthly_csv = monthly_dir / "monthly_pr_all_stations.csv"

    if monthly_csv.exists():
        continue

    if not daily_csv.exists():
        continue

    daily_df = pd.read_csv(daily_csv, parse_dates=["Date"])
    daily_df["Year"]  = daily_df["Date"].dt.year
    daily_df["Month"] = daily_df["Date"].dt.month

    monthly_df = (
        daily_df
        .groupby(["Station_ID", "Station_Name", "Basin", "Year", "Month"])["pr_mm_day"]
        .sum()
        .reset_index()
        .rename(columns={"pr_mm_day": "pr_mm_month"})
    )

    monthly_df["pr_mm_month"] = monthly_df["pr_mm_month"].round(4)
    monthly_df.to_csv(monthly_csv, index=False)

Processing models: 100%|██████████| 6/6 [00:01<00:00,  3.47it/s]


## 9. Compute Long-Term Monthly Means (Climatology)

The long-term monthly mean (LTMM) — also referred to as the **climatological mean** — is the average monthly precipitation across all years in the evaluation period (1995–2014). It characterises the seasonal precipitation cycle for each station and model, forming the basis for model evaluation against observations.

For each station, 12 LTMM values are computed (one per calendar month). Results are saved to `long_term_monthly_mean/<MODEL_NAME>/ltmm_pr_all_stations.csv` with columns: `Month`, `Station_ID`, `Station_Name`, `Basin`, `LTMM_mm_month` (long-term monthly mean, mm/month), `N_Years` (number of years contributing to the mean — useful for identifying missing-data gaps).


In [11]:
MONTH_NAMES = {
    1:"January", 2:"February", 3:"March", 4:"April",
    5:"May", 6:"June", 7:"July", 8:"August",
    9:"September", 10:"October", 11:"November", 12:"December"
}

for model in MODELS:
    monthly_csv = MONTHLY_DIR / model / "monthly_pr_all_stations.csv"
    ltmm_dir    = LTMM_DIR    / model
    ltmm_dir.mkdir(parents=True, exist_ok=True)
    ltmm_csv    = ltmm_dir    / "ltmm_pr_all_stations.csv"

    if ltmm_csv.exists():
        print(f"[SKIP] {model} — LTMM CSV already exists.")
        continue

    if not monthly_csv.exists():
        print(f"[MISSING] {model} — monthly CSV not found, skipping.")
        continue

    print(f"[LTMM] {model} ...")
    monthly_df = pd.read_csv(monthly_csv)

    ltmm_df = (
        monthly_df
        .groupby(["Station_ID", "Station_Name", "Basin", "Month"],
                 sort=True)["pr_mm_month"]
        .agg(LTMM_mm_month="mean", N_Years="count")
        .reset_index()
    )
    ltmm_df["LTMM_mm_month"] = ltmm_df["LTMM_mm_month"].round(4)
    ltmm_df["Month_Name"]    = ltmm_df["Month"].map(MONTH_NAMES)
    ltmm_df["Model"]         = model

    # Reorder columns
    ltmm_df = ltmm_df[["Model","Station_ID","Station_Name","Basin",
                         "Month","Month_Name","LTMM_mm_month","N_Years"]]
    ltmm_df.to_csv(ltmm_csv, index=False)
    print(f"  Saved → {ltmm_csv}  ({len(ltmm_df):,} rows)")

print()
print("Long-term monthly mean computation complete.")


[LTMM] CMCC-CM2-SR5 ...
  Saved → C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\long_term_monthly_mean\CMCC-CM2-SR5\ltmm_pr_all_stations.csv  (588 rows)
[LTMM] CNRM-ESM2-1 ...
  Saved → C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\long_term_monthly_mean\CNRM-ESM2-1\ltmm_pr_all_stations.csv  (588 rows)
[LTMM] EC-Earth3-Veg ...
  Saved → C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\long_term_monthly_mean\EC-Earth3-Veg\ltmm_pr_all_stations.csv  (588 rows)
[LTMM] IPSL-CM6A-LR ...
  Saved → C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\long_term_monthly_mean\IPSL-CM6A-LR\ltmm_pr_all_stations.csv  (588 rows)
[LTMM] MPI-ESM1-2-LR ...
  Saved → C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\long_term_monthly_mean\MPI-ESM1-2-LR\ltmm_pr_all_stations.csv  (588 rows)
[LTMM] NorESM2-MM ...
  Saved → C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\long_term_monthly_mean\NorESM2-MM\ltmm_pr_all_stations.csv  (588 rows)

Long-term monthly mean computation comple

## 10. Combine All Models — Summary Table

For convenience, the LTMM results from all six models are merged into a single wide-format table. Rows are stations × months; columns are model LTMM values. This format is ready for direct use in model evaluation (correlation, NSE, RMSE, MAE, PBIAS computation against observed long-term monthly means).

Saved to: `long_term_monthly_mean/ltmm_all_models_wide.csv`


In [12]:
all_ltmm = []
for model in MODELS:
    ltmm_csv = LTMM_DIR / model / "ltmm_pr_all_stations.csv"
    if not ltmm_csv.exists():
        print(f"[SKIP] {model} — LTMM CSV not found.")
        continue
    df = pd.read_csv(ltmm_csv)
    all_ltmm.append(df)

if all_ltmm:
    combined = pd.concat(all_ltmm, ignore_index=True)

    # Wide format: one column per model
    wide = combined.pivot_table(
        index=["Station_ID","Station_Name","Basin","Month","Month_Name"],
        columns="Model",
        values="LTMM_mm_month"
    ).reset_index()
    wide.columns.name = None
    wide.columns = [str(c) for c in wide.columns]

    wide_csv = LTMM_DIR / "ltmm_all_models_wide.csv"
    wide.to_csv(wide_csv, index=False)

    print(f"Combined LTMM table saved: {wide_csv}")
    print(f"Shape: {wide.shape[0]} rows × {wide.shape[1]} columns")
    print()
    print("Preview (first 6 rows):")
    print(wide.head(6).to_string(index=False))
else:
    print("No LTMM files found — run previous cells first.")


Combined LTMM table saved: C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\long_term_monthly_mean\ltmm_all_models_wide.csv
Shape: 588 rows × 11 columns

Preview (first 6 rows):
Station_ID   Station_Name   Basin  Month Month_Name  CMCC-CM2-SR5  CNRM-ESM2-1  EC-Earth3-Veg  IPSL-CM6A-LR  MPI-ESM1-2-LR  NorESM2-MM
    AB0004 KH.EL-WAHADNEH N.R.S.W      1    January       91.4819      88.2950        92.1823       72.6928        87.1949     91.9998
    AB0004 KH.EL-WAHADNEH N.R.S.W      2   February       75.9575      86.3732        80.4541       82.5618        74.6401     82.2965
    AB0004 KH.EL-WAHADNEH N.R.S.W      3      March       60.3410      75.9776        66.8313       69.7657        58.9534     57.5032
    AB0004 KH.EL-WAHADNEH N.R.S.W      4      April       22.3961      22.6636        13.1342       27.6738        19.7494     20.6731
    AB0004 KH.EL-WAHADNEH N.R.S.W      5        May        3.6632       3.1822         4.9421        7.5500         2.7862      4.1886
    AB00

## 11. Output Directory Summary

Overview of all files generated by this notebook.


In [13]:
print("=" * 72)
print("OUTPUT DIRECTORY STRUCTURE")
print("=" * 72)
for root, dirs, files in os.walk(OUTPUT_ROOT):
    # Skip hidden folders
    dirs[:] = [d for d in sorted(dirs) if not d.startswith(".")]
    level  = Path(root).relative_to(OUTPUT_ROOT).parts
    indent = "  " * len(level)
    folder = Path(root).name
    print(f"{indent}📁 {folder}/")
    subindent = "  " * (len(level) + 1)
    for f in sorted(files):
        fpath = Path(root) / f
        size  = fpath.stat().st_size / 1024
        unit  = "KB"
        if size > 1024:
            size /= 1024
            unit  = "MB"
        print(f"{subindent}📄 {f}  ({size:.1f} {unit})")

print()
print("=" * 72)
print("EXTRACTION COMPLETE")
print("=" * 72)
print()
print("Next steps:")
print("  1. Load the daily observation data (Station_Basin_Mapping.xlsx +")
print("     daily.for.stations/) for the same 1995-2014 period.")
print("  2. Aggregate observations to monthly totals.")
print("  3. Compute evaluation metrics (r, NSE, RMSE, MAE, PBIAS) by comparing")
print("     model LTMM against observed LTMM at each station.")
print("  4. Rank models per basin using the composite ranking system (Eq. 5).")


OUTPUT DIRECTORY STRUCTURE
📁 comments/
  📄 Attached standard file_ _ Revision-checklist.pdf  (96.4 KB)
  📄 Attached standard file_ comments.pdf  (69.5 KB)
  📄 station_grid_matching.csv  (4.1 KB)
  📁 codes/
    📄 01_CMIP6_Extraction_and_Aggregation.ipynb  (43.8 KB)
  📁 daily_data/
    📁 CMCC-CM2-SR5/
      📄 daily_pr_all_stations.csv  (16.9 MB)
    📁 CNRM-ESM2-1/
      📄 daily_pr_all_stations.csv  (16.9 MB)
    📁 EC-Earth3-Veg/
      📄 daily_pr_all_stations.csv  (16.9 MB)
    📁 IPSL-CM6A-LR/
      📄 daily_pr_all_stations.csv  (16.9 MB)
    📁 MPI-ESM1-2-LR/
      📄 daily_pr_all_stations.csv  (16.9 MB)
    📁 NorESM2-MM/
      📄 daily_pr_all_stations.csv  (16.9 MB)
  📁 long_term_monthly_mean/
    📄 ltmm_all_models_wide.csv  (48.8 KB)
    📁 CMCC-CM2-SR5/
      📄 ltmm_pr_all_stations.csv  (38.0 KB)
    📁 CNRM-ESM2-1/
      📄 ltmm_pr_all_stations.csv  (37.4 KB)
    📁 EC-Earth3-Veg/
      📄 ltmm_pr_all_stations.csv  (38.5 KB)
    📁 IPSL-CM6A-LR/
      📄 ltmm_pr_all_stations.csv  (38.0 KB)
    